# 02 — Feature Engineering

Construction des features ML depuis les données brutes ESP32.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")
FAKE_DATA_PATH = "../../data/raw/fake_mesures.json"


## 1. Chargement des données nettoyées

In [ ]:
from ml.src.data.loader import load_combined
from ml.src.data.cleaner import clean

df_clean = clean(load_combined(fake_path=FAKE_DATA_PATH, prefer_api=False))
print(f"Shape : {df_clean.shape}")
df_clean[["timestamp", "humidite_sol", "temperature", "humidite_air"]].head(5)


## 2. Features temporelles

On encode l'heure en sin/cos pour que 23h et 0h soient proches (cyclique).

In [ ]:
from ml.src.data.feature_builder import add_time_features

df = add_time_features(df_clean)

# Visualisation encodage cyclique
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df["hour"], df["hour_sin"], c=df["hour"], cmap="twilight", s=5, alpha=0.5)
axes[0].set_title("Heure → sin (encodage cyclique)", fontweight="bold")
axes[0].set_xlabel("Heure réelle")
axes[0].set_ylabel("sin(heure)")

axes[1].scatter(df["hour_sin"], df["hour_cos"], c=df["hour"], cmap="twilight", s=5, alpha=0.5)
axes[1].set_title("Cercle trigonométrique — heure", fontweight="bold")
axes[1].set_xlabel("sin")
axes[1].set_ylabel("cos")
axes[1].set_aspect("equal")

plt.suptitle("Encodage cyclique de l'heure", fontweight="bold")
plt.tight_layout()
plt.savefig("../../ml/reports/figures/fe_01_time_encoding.png", bbox_inches="tight")
plt.show()


## 3. Features glissantes (rolling)

Moyenne mobile et delta pour capturer les tendances récentes.

In [ ]:
from ml.src.data.feature_builder import add_rolling_features

df = add_rolling_features(df)

# Comparer humidite_sol brute vs moyennes glissantes
week = df.head(7 * 24 * 12)  # 7 jours si mesure toutes les 5min

plt.figure(figsize=(14, 5))
plt.plot(week["timestamp"], week["humidite_sol"],          label="Brute",    alpha=0.5, lw=1)
plt.plot(week["timestamp"], week["humidite_sol_mean_3"],   label="Mean 3",   lw=1.5)
plt.plot(week["timestamp"], week["humidite_sol_mean_12"],  label="Mean 12",  lw=2)
plt.title("Humidité Sol — Brute vs Moyennes Glissantes", fontweight="bold")
plt.ylabel("Humidité Sol (%)")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("../../ml/reports/figures/fe_02_rolling.png", bbox_inches="tight")
plt.show()


## 4. Construction des labels

In [ ]:
from ml.src.data.feature_builder import (
    add_classification_label,
    add_binary_label,
    add_regression_label,
)

df = add_classification_label(df)
df = add_binary_label(df)
df = add_regression_label(df, horizon=6)

print("Labels disponibles :")
print(df[["etat_sol", "besoin_eau", "humidite_prevue"]].describe())
print(f"\nRépartition etat_sol :\n{df['etat_sol'].value_counts()}")
print(f"\nBesoin eau : {df['besoin_eau'].sum()} / {len(df)} ({df['besoin_eau'].mean()*100:.1f}%)")


## 5. Dataset final

In [ ]:
from ml.src.data.feature_builder import get_feature_columns

cols = get_feature_columns()
print("Features d'entrée :")
for f in cols["features"]:
    print(f"  - {f}")

print(f"\nLabel classification : {cols['label_classification']}")
print(f"Label régression     : {cols['label_regression']}")

df_final = df.dropna(subset=["humidite_prevue"])
print(f"\nDataset final : {df_final.shape}")
df_final[cols["features"] + ["etat_sol", "besoin_eau", "humidite_prevue"]].head(5)


## 6. Corrélation features vs labels

In [ ]:
all_cols = cols["features"] + ["humidite_prevue"]
corr = df_final[all_cols].corr()["humidite_prevue"].drop("humidite_prevue").sort_values()

plt.figure(figsize=(10, 6))
colors = ["#e15759" if v < 0 else "#4e79a7" for v in corr.values]
plt.barh(corr.index, corr.values, color=colors, edgecolor="white")
plt.axvline(0, color="black", lw=0.8)
plt.title("Corrélation features → humidité_prevue (label régression)", fontweight="bold")
plt.xlabel("Corrélation de Pearson")
plt.tight_layout()
plt.savefig("../../ml/reports/figures/fe_03_feature_correlation.png", bbox_inches="tight")
plt.show()
